In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.materialized_view(
    name="shipments_gold",
    comment="Current state per shipment -- the open SCD2 row from silver"
)
def shipments_gold():
    return (
        spark.read.table("shipments_silver")
        # __END_AT is null only on the open row. Cancelled shipments have every
        # row closed, so they drop out here without needing a delete flag.
        .filter(F.col("__END_AT").isNull())
        .select(
            F.col("shipment_id").alias("ShipmentId"),
            F.col("status").alias("Status"),
            F.initcap(F.trim(F.col("location"))).alias("City"),
            F.col("customer_name").alias("CustomerName"),
            F.col("_source_updated_ts").alias("SourceUpdatedTs"),
            F.col("__START_AT").alias("ValidFromVersion"),
        )
    )

In [ ]:
@dp.materialized_view(
    name="shipments_by_city_gold",
    comment="Active shipment counts per city and status"
)
def shipments_by_city_gold():
    return (
        spark.read.table("shipments_gold")
        .filter(F.col("Status") != "DELIVERED")
        .groupBy("City", "Status")
        .agg(
            F.count("*").alias("ShipmentCount"),
            F.countDistinct("CustomerName").alias("CustomerCount"),
            F.max("SourceUpdatedTs").alias("LastUpdatedTs"),
        )
    )